In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, rand

# Create Spark session
spark = SparkSession.builder.appName("SkewFixExample").getOrCreate()

# Sample sales data
sales_data = [
    (1, "US", "A", 10),
    (2, "US", "B", 15),
    (3, "US", "A", 5),
    (4, "UK", "C", 20),
    (5, "UK", "B", 30),
    (6, "DE", "A", 50),
    (7, "DE", "C", 25),
    (8, "FR", "A", 35),
    (9, "FR", "B", 40),
    (10, "US", "A", 20)
]

# Sample product data
product_data = [
    ("A", "Product A"),
    ("B", "Product B"),
    ("C", "Product C")
]

# Define schema
sales_columns = ["transaction_id", "country", "product_id", "quantity_sold"]
product_columns = ["product_id", "product_name"]

# Create DataFrames
sales_df = spark.createDataFrame(sales_data, sales_columns)
product_df = spark.createDataFrame(product_data, product_columns)

# Fix skew by salting the country column (example for US with heavy traffic)
sales_df = sales_df.withColumn("salt", (rand() * 10).cast("int"))

# Repartition by country and salt to balance the load
sales_df = sales_df.repartition("country", "salt")

# Group by country and product_id, aggregating quantity_sold
agg_sales_df = sales_df.groupBy("country", "product_id").agg({"quantity_sold": "sum"})

# Join with product details
result_df = agg_sales_df.join(product_df, "product_id").select("country", "product_id", "product_name", "sum(quantity_sold)")

# Show the result
result_df.show()


+-------+----------+------------+------------------+
|country|product_id|product_name|sum(quantity_sold)|
+-------+----------+------------+------------------+
|     FR|         A|   Product A|                35|
|     DE|         A|   Product A|                50|
|     US|         A|   Product A|                35|
|     UK|         B|   Product B|                30|
|     FR|         B|   Product B|                40|
|     US|         B|   Product B|                15|
|     DE|         C|   Product C|                25|
|     UK|         C|   Product C|                20|
+-------+----------+------------+------------------+

